# RNN, GRU, and LSTM: Real Use Cases
This notebook demonstrates sequence models with simple, real-world-style tasks.


## 1) Why sequence models?
Sequence models handle ordered data such as text, sensor streams, and time series.
We'll compare RNN, GRU, and LSTM on small toy problems that reflect real tasks.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# Reproducibility
torch.manual_seed(42)

## 2) Helper: training loop
A small training loop to reuse across models.

In [2]:
def train_loop(model, X, y, epochs=100, lr=0.01):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if (epoch + 1) % 20 == 0:
            print(f'epoch {epoch+1}: loss={loss.item():.4f}')
    return losses

## 3) Real use case 1: sensor anomaly detection (binary classification)
We simulate a sensor sequence. The label is 1 if the last value crosses a threshold.

In [3]:
# Generate toy sensor data
batch_size = 200
seq_len = 20
features = 1

X = torch.randn(batch_size, seq_len, features)
# Make some sequences with a spike at the end
spike_mask = torch.rand(batch_size) > 0.5
X[spike_mask, -1, 0] += 4.0

y = (X[:, -1, 0] > 2.0).float().unsqueeze(1)
print('X shape:', X.shape, 'y shape:', y.shape)

X shape: torch.Size([200, 20, 1]) y shape: torch.Size([200, 1])


### 3.1 RNN model

**Note: RNN activation and outputs**
- `nn.RNN` already applies a nonlinearity internally (`tanh` by default, or `relu` if specified).
- We don’t add an extra activation on the output because the correct choice depends on the task.
- For binary classification, use `BCEWithLogitsLoss` and keep raw logits (no sigmoid in `forward`).
- For regression, keep raw outputs.
- For multiclass, use `CrossEntropyLoss` and keep raw logits (no softmax in `forward`).


In [4]:
class SensorRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1) # what is this layer for?
        # This layer maps the RNN output to the desired output shape (e.g., for regression or binary classification)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        return self.fc(out)

rnn_model = SensorRNN(input_size=1, hidden_size=16)
_ = train_loop(rnn_model, X, y, epochs=60, lr=0.01)

epoch 20: loss=0.2054
epoch 40: loss=0.0635
epoch 60: loss=0.0338


### 3.2 GRU model

In [5]:
class SensorGRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]
        return self.fc(out)

gru_model = SensorGRU(input_size=1, hidden_size=16)
_ = train_loop(gru_model, X, y, epochs=60, lr=0.01)

epoch 20: loss=0.2208
epoch 40: loss=0.0474
epoch 60: loss=0.0196


### 3.3 LSTM model

In [ ]:
class SensorLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)

lstm_model = SensorLSTM(input_size=1, hidden_size=16)
_ = train_loop(lstm_model, X, y, epochs=60, lr=0.01)

## 4) Real use case 2: sequence regression (temperature forecast)
Given a short sequence of temperatures, predict the next value.

In [6]:
# Create a smooth temperature-like signal
t = torch.linspace(0, 50, steps=300)
signal = torch.sin(t * 0.2) + 0.1 * torch.randn_like(t)

seq_len = 15
X_reg = []
y_reg = []
for i in range(len(signal) - seq_len):
    X_reg.append(signal[i:i+seq_len])
    y_reg.append(signal[i+seq_len])
X_reg = torch.stack(X_reg).unsqueeze(-1)
y_reg = torch.stack(y_reg).unsqueeze(-1)

print('X_reg:', X_reg.shape, 'y_reg:', y_reg.shape)

X_reg: torch.Size([285, 15, 1]) y_reg: torch.Size([285, 1])


In [7]:
class ForecastLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)

model = ForecastLSTM(input_size=1, hidden_size=32)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    optimizer.zero_grad()
    preds = model(X_reg)
    loss = criterion(preds, y_reg)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 25 == 0:
        print(f'epoch {epoch+1}: loss={loss.item():.4f}')

epoch 25: loss=0.0196
epoch 50: loss=0.0140
epoch 75: loss=0.0136
epoch 100: loss=0.0132


## 5) Real use case 3: text-like sequence classification (toy)
We simulate sequences of token IDs. Label is 1 if a special token appears.

In [ ]:
vocab_size = 20
seq_len = 12
batch_size = 256
special_token = 7

X_text = torch.randint(0, vocab_size, (batch_size, seq_len))
y_text = (X_text == special_token).any(dim=1).float().unsqueeze(1)

class TextGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.embed(x)
        out, _ = self.gru(x)
        out = out[:, -1, :]
        return self.fc(out)

text_model = TextGRU(vocab_size, embed_dim=8, hidden_size=16)
_ = train_loop(text_model, X_text, y_text, epochs=60, lr=0.01)

## 6) Summary
- RNN: simplest, can struggle with long sequences.
- GRU: fewer gates than LSTM, often faster.
- LSTM: strongest for longer dependencies.


## 7) Assignments
1) Increase sequence length to 50 and compare RNN vs LSTM.
2) Add dropout to GRU/LSTM and observe training loss.
3) For the text task, change the rule: label 1 if two specific tokens appear in order.
4) Change the RNN nonlinearity to ReLU (`nn.RNN(..., nonlinearity='relu')`) and compare loss.
5) Replace the last-timestep pooling with mean pooling over time; compare results.
6) Add gradient clipping (`torch.nn.utils.clip_grad_norm_`) and observe stability.
